In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from datetime import datetime, timedelta

from core.ml.price.backtest import BacktestEngine
from core.portfolio.asset import Portfolio, Asset
from core.data.instrument.provider import InstrumentProvider
from core.data.instrument.instrument import Instrument
from core.ml.price.modelalt import PriceDirPredictorRF, PriceDirPredictorLGBM, PriceDirPredictorXGB, PriceDirPredictorStacked
from core.ml.price.strategy import ATRStopLossStrategy

ip = InstrumentProvider()
instrument: Instrument = ip.get_instrument('MSFT')

predictor = PriceDirPredictorLGBM(instrument=instrument, horizon_days=5)
strategy = ATRStopLossStrategy(predictor)

portfolio = Portfolio('BT1', 'USD')
start_date = datetime(2026, 1, 1)
engine = BacktestEngine(portfolio, start_date, strategy, initial_cash=1000.0)


print("--- Backtest ---")
reached_present = False
i = 0
while not reached_present:
    reached_present = engine.next_day()
    print(f"Day {i+1} ({engine.date.strftime('%Y-%m-%d')}): Assets Value: ${engine.assets_values:,.2f}, Cash: ${engine.cash}, Total Value: ${engine.total_value}")
    i += 1

engine.show_transaction_log_df()

--- Backtest ---
Day 1 (2026-01-02): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 2 (2026-01-03): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 3 (2026-01-04): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 4 (2026-01-05): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 5 (2026-01-06): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 6 (2026-01-07): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 7 (2026-01-08): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 8 (2026-01-09): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 9 (2026-01-10): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 10 (2026-01-11): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 11 (2026-01-12): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 12 (2026-01-13): Assets Value: $0.00, Cash: $1000.0, Total Value: $1000.0
Day 13 (2026-01-14): Assets Value: $0.00, Cash: $1000.0,

Type,Symbol,Volume,Price ($),Total Amount ($),Fee ($)
BUY,MSFT,0.792896,478.45,379.36,0.00
SELL,MSFT,0.792892,431.58,342.20,0.00
BUY,MSFT,0.785723,410.33,322.41,0.00
SELL,MSFT,0.785721,406.90,319.71,0.00
BUY,MSFT,0.695491,413.07,287.29,0.00
SELL,MSFT,0.695491,414.22,288.09,0.00


In [2]:
portfolio2 = Portfolio('BT2', 'USD')
start_date = datetime(2025, 1, 1)
market_data = instrument.get_market_data_at_closest_trading_day(start_date)
price = portfolio2.convert_to_native_currency(market_data['close'], instrument.currency)

volume = 1000 / price if price != 0 else 0

asset = Asset(instrument, volume, price, start_date)

portfolio2.add(asset)

portfolio2.value

1950.4666188083274